In [3]:
# model_checkpoint = "facebook/esm2_t33_650M_UR50D"
# model_checkpoint = "facebook/esm2_t12_35M_UR50D"
model_checkpoint = "facebook/esm2_t30_150M_UR50D" # batch size 4, 20GB mem, 

In [1]:

import pandas as pd

df = pd.read_csv("./data/training_dataset_less1023.csv")

In [7]:

# label 0: non SPTM_CM
# label 1: SPTM_CM
# label 2: strong SPTM_CM, validated by experimental data

label0_sequences = df.loc[df["label"]==0," sequenceValue"].tolist()
label0_labels = [0 for protein in label0_sequences]

label1_sequences = df.loc[df["label"]==1," sequenceValue"].tolist()
label1_labels = [1 for protein in label1_sequences]

label2_sequences = df.loc[df["label"]==2," sequenceValue"].tolist()
label2_labels = [2 for protein in label2_sequences]

In [8]:


sequences = label0_sequences + label1_sequences + label2_sequences
labels = label0_labels + label1_labels + label2_labels

# Quick check to make sure we got it right
len(sequences) == len(labels)

True

In [9]:


from sklearn.model_selection import train_test_split

train_sequences, test_sequences, train_labels, test_labels = train_test_split(sequences, labels, test_size=0.20, shuffle=True)

In [10]:


from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [12]:

train_tokenized = tokenizer(train_sequences)
test_tokenized = tokenizer(test_sequences)

In [13]:


from datasets import Dataset
train_dataset = Dataset.from_dict(train_tokenized)
test_dataset = Dataset.from_dict(test_tokenized)

train_dataset

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 3362
})

In [14]:


train_dataset = train_dataset.add_column("labels", train_labels)
test_dataset = test_dataset.add_column("labels", test_labels)
train_dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3362
})

In [16]:
import torch
import torch.distributed as dist
import torch.nn as nn
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

num_labels = 3  # Add 1 since 0 can be a label

# load model from local directory, esm2_t12_35M_UR50D-SPTM_CM_35M_5epochs_split5
# model = AutoModelForSequenceClassification.from_pretrained("./esm2_t12_35M_UR50D-SPTM_CM_35M_5epochs_split5/checkpoint-1996", num_labels=num_labels)
# model = AutoModelForSequenceClassification.from_pretrained("./esm2_t12_35M_UR50D-SPTM_CM_35M_split5epoch2_split20_3epochs/checkpoint-998", num_labels=num_labels)
model = AutoModelForSequenceClassification.from_pretrained("./esm2_t30_150M_UR50D-SPTM_CM_150M_5epochs/checkpoint-1682", num_labels=num_labels)

print("Let's use", torch.cuda.device_count(), "GPUs!")

#  model = nn.DataParallel(model)

/home/zf77/.conda/envs/torch/lib/python3.11/site-packages/torch/cuda/__init__.py:611: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Some weights of EsmForSequenceClassification were not initialized from the model checkpoint at facebook/esm2_t12_35M_UR50D and are newly initialized: ['classifier.dense.bias', 'classifier.out_proj.bias', 'classifier.out_proj.weight', 'classifier.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [18]:

# model_checkpoint = "facebook/esm2_t33_650M_UR50D"

model_name = model_checkpoint.split("/")[-1]
batch_size = 2

args = TrainingArguments(
    f"{model_name}-SPTM_CM_split20epoch2_split20_3epochs_1GPU_test",
    evaluation_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy"
#    deepspeed="ds_config_zero3.json"
#    push_to_hub=True,
)

In [19]:


from evaluate import load
import numpy as np

metric = load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)

In [91]:


trainer = Trainer(
    model,
    args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

In [1]:
trainer.train() 

